In [23]:
import tensorflow as tf
from tensorflow.keras.preprocessing import image_dataset_from_directory
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

BASE_DIR = "dataset"
IMG_SIZE = (224, 224)
BATCH_SIZE = 8

# Derive training dataset
train_ds = image_dataset_from_directory(
    BASE_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=True,
    validation_split=0.2,
    subset="training",
    seed=67
)

# Derive validation dataset
val_ds = image_dataset_from_directory(
    BASE_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    validation_split=0.2,
    subset="validation",
    seed=67
)

# Save class names
# Preprocessing gets rid of them in dataset metadata
class_names = train_ds.class_names

# Preprocess x (images) in dataset to MobileNetV2 model
train_ds = train_ds.map(lambda x, y: (preprocess_input(x), y))
val_ds = val_ds.map(lambda x, y: (preprocess_input(x), y))

Found 788 files belonging to 4 classes.
Using 631 files for training.
Found 788 files belonging to 4 classes.
Using 157 files for validation.


In [24]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers, models

# Load MobileNetV2 (no top layer)
base_model = MobileNetV2(input_shape=(224, 224, 3),
                         include_top=False,
                         weights='imagenet')

# Freeze convolutional base
base_model.trainable = False

# Functional instead of sequential (better serialization when saving)
inputs = tf.keras.Input(shape=(224, 224, 3)) # Input tensor
x = base_model(inputs, training=False) # Pass input through base model (high-level features)
x = layers.GlobalAveragePooling2D()(x) # Convert feature map to single vector per image
x = layers.Dense(128, activation='relu')(x) # Add layer to learn abstract representations
x = layers.Dropout(0.3)(x) # Dropout for regularization (prevent overfitting)
outputs = layers.Dense(len(class_names), activation='softmax')(x) # Output layer for classification

model = tf.keras.Model(inputs, outputs)

model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

model.summary()

Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_7 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_3      │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 128)            │       163,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 4)              │           516 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,422,468 (9.24 MB)

 Trainable params: 164,484 (642.52 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [25]:
# Train model (5 epochs)
history = model.fit(train_ds,
                    validation_data=val_ds,
                    epochs=8)

Epoch 1/8
79/79 ━━━━━━━━━━━━━━━━━━━━ 23s 172ms/step - accuracy: 0.8526 - loss: 0.3999 - val_accuracy: 0.9873 - val_loss: 0.0936
Epoch 2/8
79/79 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - accuracy: 0.9762 - loss: 0.0731 - val_accuracy: 0.9809 - val_loss: 0.0534
Epoch 3/8
79/79 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - accuracy: 0.9952 - loss: 0.0255 - val_accuracy: 0.9936 - val_loss: 0.0373
Epoch 4/8
79/79 ━━━━━━━━━━━━━━━━━━━━ 2s 20ms/step - accuracy: 0.9952 - loss: 0.0210 - val_accuracy: 0.9873 - val_loss: 0.0389
Epoch 5/8
79/79 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - accuracy: 0.9984 - loss: 0.0125 - val_accuracy: 0.9936 - val_loss: 0.0368
Epoch 6/8
79/79 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 1.0000 - loss: 0.0097 - val_accuracy: 0.9873 - val_loss: 0.0265
Epoch 7/8
79/79 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 1.0000 - loss: 0.0064 - val_accuracy: 0.9936 - val_loss: 0.0260
Epoch 8/8
79/79 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.9984 - loss: 0.0055 - val_accuracy: 0.9936 - val_los

In [26]:
import numpy as np
from tensorflow.keras.utils import load_img, img_to_array

# Test model prediction
img_path = "dataset/neutral/100.jpg"
img = load_img(img_path, target_size=IMG_SIZE)
x = img_to_array(img) # Convert image to tensor
x = preprocess_input(x) # Preprocess for MobileNetV2
x = np.expand_dims(x, axis=0) # Add batch dimension (model expects batches)

pred = model.predict(x)
print("Raw predictions:", pred)
print("Predicted class:", class_names[np.argmax(pred)]) # Index of highest probability

1/1 ━━━━━━━━━━━━━━━━━━━━ 4s 4s/step
Raw predictions: [[5.0001422e-06 9.9978161e-01 1.3542415e-04 7.7992227e-05]]
Predicted class: neutral


In [27]:
model.save('nailong_exp_model.keras')